# COSMOS QA: measure before building
The rules are fixed in Cell 2 before any data is loaded; Cell 8 applies them automatically. Nothing is scored or published.

## Cell 1 — Install and imports
No GPU needed. Nothing is scored or published by this notebook.

In [ ]:
!pip install -q datasets "transformers>=4.48"
import re, random
from collections import Counter
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer

## Cell 2 — Settings and the rules, fixed before looking
Every threshold is written down here, before any data is loaded.

In [ ]:
DATASET = "allenai/cosmos_qa"          # the Allen Institute's copy; license confirmed by the authors (see card)
STUDENT = "answerdotai/ModernBERT-base"
MAX_LEN = 512
TEMPLATE_TOKENS = 40                   # room for [CLS], [SEP]s, 4 markers and a short instruction
SEED = 0

RULES = {
    "longest option accuracy":              ("<=", 0.35),
    "most-overlap-with-passage accuracy":   ("<=", 0.35),
    "smallest answer-position share":       (">=", 0.15),
    "largest answer-position share":        ("<=", 0.35),
    "'None of the above' right when shown": ("between", (0.10, 0.40)),
    "share that fits in 512 tokens":        (">=", 0.95),
    "training rows":                        (">=", 3000),
    "validation rows":                      (">=", 600),
}
RESULTS = {}

## Cell 3 — Load and look at the structure
Columns, sizes, one example, and whether the test split has labels.

In [ ]:
def load_cosmos():
    """1) Hugging Face's Parquet copy of the dataset; 2) if that fails, the authors' own files on GitHub."""
    for kwargs in ({"name": "default"}, {}):
        try:
            d = load_dataset(DATASET, revision="refs/convert/parquet", **kwargs)
            print("loaded Hugging Face's Parquet copy", kwargs)
            return d
        except Exception as e:
            print(f"Parquet copy {kwargs} failed: {type(e).__name__}: {str(e)[:120]}")
    from datasets import Dataset, DatasetDict
    base = "https://github.com/wilburOne/cosmosqa/raw/master/data/"      # where the old script downloaded from
    parts = {"train": pd.read_csv(base + "train.csv"), "validation": pd.read_csv(base + "valid.csv")}
    print("loaded the authors' files from GitHub (train.csv, valid.csv)")
    return DatasetDict({k: Dataset.from_pandas(v, preserve_index=False) for k, v in parts.items()})

ds = load_cosmos()
print(ds)
ex = ds["train"][0]
print("\none example:")
for k, v in ex.items():
    print(f"  {k}: {str(v)[:300]}")
test_labels = Counter(ds["test"]["label"]) if "test" in ds else Counter()
print("\ntest split labels:", dict(test_labels.most_common(5)),
      "-> hidden (all the same placeholder)" if len(test_labels) == 1 else "")

def rows_of(split):
    out = []
    for e in ds[split]:
        opts = [e[f"answer{i}"] for i in range(4)]
        if e["label"] not in (0, 1, 2, 3) or any(not (o or "").strip() for o in opts):
            continue
        out.append(dict(context=e["context"], question=e["question"], options=opts, label=int(e["label"])))
    return out

ROWS = {s: rows_of(s) for s in ("train", "validation")}
RESULTS["training rows"] = len(ROWS["train"])
RESULTS["validation rows"] = len(ROWS["validation"])
print({s: len(r) for s, r in ROWS.items()}, "usable rows (labelled, four non-empty options)")

default/train/0000.parquet: reconstructing file:   0%|          |  0.00B / 7.92MB            

default/train/0000.parquet: downloading bytes:           |  0.00B            

default/validation/0000.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

default/validation/0000.parquet: downloading bytes:           |  0.00B            

default/test/0000.parquet: reconstructing file:   0%|          |  0.00B / 2.87MB            

default/test/0000.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

loaded Hugging Face's Parquet copy {'name': 'default'}
DatasetDict({
    train: Dataset({
        features: ['id', 'context', 'question', 'answer0', 'answer1', 'answer2', 'answer3', 'label'],
        num_rows: 25262
    })
    validation: Dataset({
        features: ['id', 'context', 'question', 'answer0', 'answer1', 'answer2', 'answer3', 'label'],
        num_rows: 2985
    })
    test: Dataset({
        features: ['id', 'context', 'question', 'answer0', 'answer1', 'answer2', 'answer3', 'label'],
        num_rows: 6963
    })
})

one example:
  id: 3Q9SPIIRWJKVQ8244310E8TUS6YWAC##34V1S5K3GTZMDUBNBIGY93FLDOB690##A1S1K7134S2VUC##Blog_1044056##q1_a1##3XU9MCX6VQQG7YPLCSAFDPQNH4GR20
  context: Good Old War and person L : I saw both of these bands Wednesday night , and they both blew me away . seriously . Good Old War is acoustic and makes me smile . I really can not help but be happy when I listen to them ; I think it 's the fact that they seemed so happy themselves when they played .
  qu

## Cell 4 — Answer position
Is the right answer spread evenly over the four positions?

In [ ]:
pos = pd.DataFrame({s: pd.Series([r["label"] for r in rows]).value_counts(normalize=True).sort_index()
                    for s, rows in ROWS.items()})
pos.index = [f"position {i + 1}" for i in pos.index]
print(pos.round(3))
RESULTS["smallest answer-position share"] = float(pos["train"].min())
RESULTS["largest answer-position share"] = float(pos["train"].max())

            train  validation
position 1  0.251       0.249
position 2  0.250       0.244
position 3  0.250       0.255
position 4  0.248       0.252


## Cell 5 — 'None of the above'
How often it appears, and how often it's the right answer when it does.

In [ ]:
NOTA = re.compile(r"none of the above", re.I)
for split, rows in ROWS.items():
    with_nota = [r for r in rows if any(NOTA.search(o) for o in r["options"])]
    right = [r for r in with_nota if NOTA.search(r["options"][r["label"]])]
    share_shown, share_right = len(with_nota) / len(rows), len(right) / max(1, len(with_nota))
    print(f"{split}: shown in {share_shown:.1%} of questions; right answer in {share_right:.1%} of those")
    if split == "train":
        RESULTS["'None of the above' right when shown"] = share_right

train: shown in 75.5% of questions; right answer in 7.8% of those
validation: shown in 73.3% of questions; right answer in 11.8% of those


## Cell 6 — Can the answer be picked without really reading?
Two simple tricks. If either works much better than chance (25%), the options leak the answer.

In [ ]:
WORD = re.compile(r"[a-z']+")
STOP = set("a an the of to and or in on at for is was are were be been it its this that with as by he she they i you we".split())
def words(t):
    return {w for w in WORD.findall(t.lower()) if w not in STOP}

def credit(scores, gold):
    # Ties are split evenly, so a trick never gets credit just from picking the first option
    best = max(scores)
    tied = [i for i, s in enumerate(scores) if s == best]
    return (1 / len(tied)) if gold in tied else 0.0

for split, rows in ROWS.items():
    longest = np.mean([credit([len(o) for o in r["options"]], r["label"]) for r in rows])
    overlap = np.mean([credit([len(words(o) & words(r["context"])) for o in r["options"]], r["label"]) for r in rows])
    print(f"{split}: pick the longest option = {longest:.3f} | pick the most overlap with the passage = {overlap:.3f} "
          f"| chance = 0.250")
    if split == "train":
        RESULTS["longest option accuracy"] = float(longest)
        RESULTS["most-overlap-with-passage accuracy"] = float(overlap)

train: pick the longest option = 0.288 | pick the most overlap with the passage = 0.285 | chance = 0.250
validation: pick the longest option = 0.296 | pick the most overlap with the passage = 0.164 | chance = 0.250


## Cell 7 — Does it fit the student?
Passage + question + four options, in ModernBERT tokens.

In [ ]:
tok = AutoTokenizer.from_pretrained(STUDENT)
sample = random.Random(SEED).sample(ROWS["train"], min(3000, len(ROWS["train"])))
lens = np.array([len(tok(r["context"] + " " + r["question"] + " " + " ".join(r["options"]),
                         add_special_tokens=False)["input_ids"]) + TEMPLATE_TOKENS for r in sample])
print(f"tokens per question (sample of {len(sample)}): median {np.median(lens):.0f}, "
      f"90% below {np.quantile(lens, 0.9):.0f}, 99% below {np.quantile(lens, 0.99):.0f}, longest {lens.max()}")
RESULTS["share that fits in 512 tokens"] = float((lens <= MAX_LEN).mean())
print(f"share within {MAX_LEN}: {RESULTS['share that fits in 512 tokens']:.1%}")

print("\nthree example questions:")
for r in random.Random(SEED + 1).sample(ROWS["train"], 3):
    print("-" * 70)
    print("PASSAGE:", r["context"][:500])
    print("QUESTION:", r["question"])
    for i, o in enumerate(r["options"]):
        print(f"  {'*' if i == r['label'] else ' '} {chr(65 + i)}. {o}")

config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

tokens per question (sample of 3000): median 153, 90% below 196, 99% below 241, longest 285
share within 512: 100.0%

three example questions:
----------------------------------------------------------------------
PASSAGE: I really wanted to post this before I sleep , but the computer was hogged > _ < and shut down afterwards . Because yesterday 's olympics was so good that I stayed up until pretty late . I was enjoying the badminton doubles yesterday .
QUESTION: Is badminton a sport featured in the olympics ?
  * A. Its actually featured in summer olympics .
    B. None of the above choices .
    C. Badminton is really not a sport , so it ca nt be featured in the olympics .
    D. Badminton doubles are not featured in the summer or any other onlympic games .
----------------------------------------------------------------------
PASSAGE: Yep ... that 's me ! We were supposed to shoot my final film tomorrow ... and guess what . The doctor of the medical center where I ' m shooting calls

## Cell 8 — The verdict, applied automatically
Each rule from Cell 2, checked against the measured value.

In [ ]:
def passes(kind, threshold, value):
    if kind == "<=":
        return value <= threshold
    if kind == ">=":
        return value >= threshold
    lo, hi = threshold
    return lo <= value <= hi

verdict = []
for name, (kind, threshold) in RULES.items():
    value = RESULTS[name]
    verdict.append({"rule": name, "needs": f"{kind} {threshold}",
                    "measured": f"{value:.3f}" if isinstance(value, float) else str(value),
                    "result": "PASS" if passes(kind, threshold, value) else "FAIL"})
verdict = pd.DataFrame(verdict)
print(verdict.to_string(index=False))
print("\nOVERALL:", "PASS: COSMOS QA can be built into v2.2" if (verdict.result == "PASS").all()
      else "FAIL: don't build it as is")

                                rule              needs measured result
             longest option accuracy            <= 0.35    0.288   PASS
  most-overlap-with-passage accuracy            <= 0.35    0.285   PASS
      smallest answer-position share            >= 0.15    0.248   PASS
       largest answer-position share            <= 0.35    0.251   PASS
'None of the above' right when shown between (0.1, 0.4)    0.078   FAIL
       share that fits in 512 tokens            >= 0.95    1.000   PASS
                       training rows            >= 3000    25262   PASS
                     validation rows             >= 600     2985   PASS

OVERALL: FAIL: don't build it as is
